# Add metadata to an existing experiment

## Imports

In [64]:
import psycopg2
import pandas as pd

# import a folder in the parent directory
import sys
sys.path.append('../')
import dbtools.dbtools as dbt
import dbtools.load as load

## Connection

In [65]:
try:
    # Connect to the PostgreSQL database
    conn = dbt.connect()
    print("Connected to the database")

except (Exception, psycopg2.DatabaseError) as error:
    print(error)

Connected to the database


## Experiment path

Enter the `folder_path` of the experiment you want to attach metadata to. This must match the `folder_path` already stored in the `experiments` table.

In [66]:
experiment_path = r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/05_Models/Hexcel/2026/CNN/IQ/2025Baseline'

## Check the experiment exists

Looks up `experiment_path` in the `experiments` table. If found, its current metadata is shown below so you can see what's already there before adding more.

In [67]:
experiment_id = None

try:
    experiment_id = int(dbt.get_id('experiments', ['folder_path_experiment'], [experiment_path]))
    print(f"Experiment found in database (id={experiment_id})")

    cursor = conn.cursor()
    cursor.execute(
        "SELECT key, value, type FROM experiment_metadata WHERE experiment_id = %s ORDER BY key",
        (experiment_id,)
    )
    columns = [desc[0] for desc in cursor.description]
    existing_metadata = pd.DataFrame(cursor.fetchall(), columns=columns)
    cursor.close()
    conn.rollback()  # read-only lookup; release the transaction so conn is idle again

    if existing_metadata.empty:
        print("No metadata stored yet for this experiment.")
    else:
        display(existing_metadata)

except ValueError as e:
    print(f"Experiment NOT found in database: {e}")

Experiment found in database (id=55)


,key,value,type
0,author,Alberto Vicente del Egido,string


## Metadata to add

Each entry needs a `key`, `value`, and `type`. If a `key` already exists for this experiment, its `value`/`type` will be overwritten.

In [68]:
metadata = [
    {'key': 'Sample Filter', 'value': 'Na', 'type': 'string'},
    {'key': 'Panels To Exclude', 'value': 'Na_02', 'type': 'string'},
    {'key': 'Max amplitude threshold', 'value': 60, 'type': 'integer'},
    {'key': 'UT alignment point', 'value': 50, 'type': 'integer'},
    {'key': 'Crop start', 'value': 0, 'type': 'integer'},
    {'key': 'Crop end', 'value': 500, 'type': 'integer'},
    {'key': 'ut patch', 'value': 1, 'type': 'integer'},
    {'key': 'xct patch', 'value': 7, 'type': 'integer'},
    {'key': 'folds', 'value': 4, 'type': 'integer'},
    {'key': 'fold strategy', 'value': '1 test panel, 2 panel per fold, 1 fold for validation', 'type': 'string'},
    {'key': 'augmentation', 'value': 'Yes', 'type': 'string'}
]

## Add the metadata

Only runs if the experiment was found above.

In [69]:
if experiment_id is None:
    print("Experiment was not found above, aborting. Fix experiment_path and re-run the check cell.")
else:
    load.add_experiment_metadata(conn, experiment_path, metadata)

Added 11 metadata entries to experiment ID 55
